# WildFusion similarities

Run one external dataset and representation at a time. Set APEXID_DATASET_NAME, APEXID_GLOBAL_DESCRIPTOR, and APEXID_MODEL_CONFIG. Set APEXID_WEIGHTS for a trained checkpoint. The open-set roles come from 03_open_set_roles.ipynb. Set APEXID_CALIBRATION_COHORT to reuse the recorded 512-image calibration sample.

In [ ]:
import json
import os
import sys
from pathlib import Path

REPO_ROOT = Path(os.environ.get("APEXID_REPO_ROOT", Path.cwd())).resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "apexid").is_dir():
    raise FileNotFoundError("Start the notebook in the repository or set APEXID_REPO_ROOT.")
sys.path.insert(0, str(REPO_ROOT))

import hashlib
import numpy as np
import pandas as pd
import torch
import torchvision.transforms as T
from PIL import Image
from wildlife_datasets.datasets import WildlifeDataset
from wildlife_tools.features import DeepFeatures
from wildlife_tools.features.local import AlikedExtractor, DiskExtractor
from wildlife_tools.similarity import CosineSimilarity
from wildlife_tools.similarity.calibration import IsotonicCalibration
from wildlife_tools.similarity.pairwise.lightglue import MatchLightGlue
from wildlife_tools.similarity.wildfusion import SimilarityPipeline, WildFusion

if not torch.cuda.is_available():
    raise RuntimeError("WildFusion extraction and LightGlue require CUDA for this workflow.")

In [ ]:
ALLOWED_DATASETS = ("bristol_gorillas", "elephants", "gcn_id", "jaguar_antonio")
DATASET_NAME = os.environ["APEXID_DATASET_NAME"]
GLOBAL_DESCRIPTOR = os.environ["APEXID_GLOBAL_DESCRIPTOR"]
MODEL_CONFIG = Path(os.environ["APEXID_MODEL_CONFIG"])
WEIGHTS = Path(os.environ["APEXID_WEIGHTS"]) if os.environ.get("APEXID_WEIGHTS") else None
if DATASET_NAME not in ALLOWED_DATASETS:
    raise ValueError(f"Choose a dataset from {ALLOWED_DATASETS}")
if Path(GLOBAL_DESCRIPTOR).name != GLOBAL_DESCRIPTOR or GLOBAL_DESCRIPTOR in {"", ".", ".."}:
    raise ValueError("APEXID_GLOBAL_DESCRIPTOR must be a single directory name")

STAGE00_ROOT = Path(os.environ.get("APEXID_PREPARED_INPUT", REPO_ROOT / "outputs" / "apexid_cvpr_prepared"))
SPLITS_ROOT = Path(os.environ.get("APEXID_STAGE03_SPLITS", REPO_ROOT / "outputs" / "stage03" / "03_splits"))
OUTPUT_PARENT_ROOT = Path(os.environ.get("APEXID_STAGE03A_OUTPUT", REPO_ROOT / "outputs" / "stage03" / "03a"))
OUTPUT_ROOT = OUTPUT_PARENT_ROOT / GLOBAL_DESCRIPTOR / DATASET_NAME
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
B = 256
GLOBAL_BATCH_SIZE = LOCAL_BATCH_SIZE = 16
CALIBRATION_SIZE = 512
CALIBRATION_SEED = 42
CALIBRATION_SAMPLE_NAME = "wildfusion_calibration_sample_512.csv"
DEVICE = torch.device("cuda")

In [ ]:
# Load frozen manifests and resolve their original paths to the image mount.
PREPARATION_MANIFEST = json.loads((STAGE00_ROOT / "preparation_manifest.json").read_text(encoding="utf-8"))
if PREPARATION_MANIFEST.get("status") != "passed":
    raise RuntimeError("Stage 00 preparation_manifest.json is not marked passed")

MANIFEST_PATHS = {
    "apexid_calibration": STAGE00_ROOT / "manifests" / "apexid_calibration_manifest.csv",
    **{name: STAGE00_ROOT / "manifests" / f"{name}_canonical_manifest.csv" for name in ALLOWED_DATASETS},
}
from apexid.evaluation.embeddings import read_path_map, remap_path

PATH_MAP = read_path_map(os.environ.get("APEXID_PATH_MAP"))
IMAGE_ROOT = Path(os.environ.get("APEXID_IMAGE_ROOT", REPO_ROOT))

def resolve_image_path(frozen_path: str) -> Path:
    path = Path(remap_path(frozen_path, PATH_MAP))
    path = path if path.is_absolute() else IMAGE_ROOT / path
    if not path.is_file():
        raise FileNotFoundError(f"Mount {frozen_path} or set APEXID_PATH_MAP.")
    return path.resolve()


frames = {}
for dataset_name, path in MANIFEST_PATHS.items():
    if not path.is_file():
        raise FileNotFoundError(path)
    frame = pd.read_csv(path, dtype=str).fillna("")
    required = {"image_id", "path", "identity", "dataset"}
    if not required <= set(frame.columns):
        raise ValueError(f"{dataset_name}: missing columns {sorted(required - set(frame.columns))}")
    if frame["image_id"].duplicated().any():
        raise ValueError(f"{dataset_name}: duplicate image_id")
    frames[dataset_name] = frame.reset_index(drop=True)

def catalogue_frame(frame: pd.DataFrame, *, namespace_identity: bool) -> pd.DataFrame:
    out = frame[["image_id", "path", "identity", "dataset"]].copy()
    out["frozen_path"] = out["path"].astype(str)
    if namespace_identity:
        out["identity"] = out["dataset"].astype(str) + "::" + out["identity"].astype(str)
    return out.reset_index(drop=True)


def resolve_catalogue_paths(frame: pd.DataFrame, label: str) -> pd.DataFrame:
    out = frame.copy()
    print(f"Resolving {len(out)} {label} image paths ...", flush=True)
    out["path"] = [str(resolve_image_path(path)) for path in out["frozen_path"]]
    print(f"Resolved {label}", flush=True)
    return out


full_calibration_catalogue = pd.concat(
    [catalogue_frame(frame, namespace_identity=True) for frame in frames.values()],
    ignore_index=True,
)
selected_unresolved = catalogue_frame(frames[DATASET_NAME], namespace_identity=False)

SPLIT_PATH = SPLITS_ROOT / DATASET_NAME / "open_role_split.csv"
split_frame = pd.read_csv(SPLIT_PATH, dtype=str).fillna("")
required_split = {"image_id", "identity", "open_role"}
if not required_split <= set(split_frame.columns):
    raise ValueError(f"{SPLIT_PATH} lacks {sorted(required_split - set(split_frame.columns))}")
if split_frame["image_id"].duplicated().any():
    raise ValueError(f"{SPLIT_PATH}: duplicate image_id")
valid_roles = {"gallery", "query_known", "query_novel"}
if not set(split_frame["open_role"]).issubset(valid_roles):
    raise ValueError(f"{SPLIT_PATH}: unexpected open_role values")
has_gallery = split_frame["open_role"].eq("gallery").any()
PROTOCOL = "gallery_attach_optuna" if has_gallery else "query_only_defaults"
if PROTOCOL == "query_only_defaults":
    if (split_frame["open_role"] != "query_novel").any():
        raise RuntimeError("Gallery-free split must be 100% query_novel")
else:
    if not {"gallery", "query_known", "query_novel"} <= set(split_frame["open_role"]):
        raise RuntimeError("Standard split must include gallery, query_known, and query_novel")
print("PROTOCOL from split:", PROTOCOL)

catalogue_ids = set(selected_unresolved["image_id"].astype(str))
split_ids = split_frame["image_id"].astype(str)
missing = set(split_ids) - catalogue_ids
if missing:
    raise KeyError(f"{len(missing)} split image_ids are missing from the Stage 00 canonical manifest")
keep_columns = ["image_id", "open_role"] + [
    column for column in ("observation_group_id", "grouping_method", "split_seed") if column in split_frame.columns
]
order = {image_id: index for index, image_id in enumerate(split_ids)}
selected_unresolved = selected_unresolved.merge(split_frame[keep_columns], on="image_id", how="inner")
selected_unresolved["_ord"] = selected_unresolved["image_id"].map(order)
selected_unresolved = selected_unresolved.sort_values("_ord").drop(columns=["_ord"]).reset_index(drop=True)
print("SPLIT_PATH:", SPLIT_PATH)
print(selected_unresolved.groupby("open_role").size())
print("Canonical images excluded from split:", len(catalogue_ids) - len(selected_unresolved))


# This notebook creates the 512-image cohort. It is not a Stage 00 input.
# Seed 42 + frozen manifests make the same membership on every animal/model run.
def deterministic_stratified_sample(catalogue, size, seed):
    if size > len(catalogue):
        raise ValueError(f"CALIBRATION_SIZE={size} exceeds catalogue size {len(catalogue)}")
    rng = np.random.default_rng(seed)
    dataset_names = sorted(catalogue["dataset"].astype(str).unique())
    base, remainder = divmod(size, len(dataset_names))
    selected_indices = []
    for rank, dataset_name in enumerate(dataset_names):
        pool = catalogue.index[catalogue["dataset"].eq(dataset_name)].to_numpy()
        take = min(len(pool), base + int(rank < remainder))
        selected_indices.extend(rng.choice(pool, size=take, replace=False).tolist())
    if len(selected_indices) < size:
        remaining = np.setdiff1d(catalogue.index.to_numpy(), np.asarray(selected_indices), assume_unique=False)
        selected_indices.extend(rng.choice(remaining, size=size - len(selected_indices), replace=False).tolist())
    sampled = catalogue.loc[selected_indices].copy()
    return sampled.sort_values(["dataset", "image_id"], kind="stable").reset_index(drop=True)

print(f"Creating {CALIBRATION_SIZE}-image calibration cohort from Stage 00 manifests (seed={CALIBRATION_SEED})")
frozen_cohort = os.environ.get("APEXID_CALIBRATION_COHORT")
if frozen_cohort:
    membership = pd.read_csv(frozen_cohort, dtype=str)[["dataset", "image_id"]]
    calibration_catalogue = membership.merge(
        full_calibration_catalogue, on=["dataset", "image_id"], how="left", validate="one_to_one"
    )
    if calibration_catalogue["frozen_path"].isna().any():
        raise ValueError("Calibration cohort contains unknown image IDs")
else:
    calibration_catalogue = deterministic_stratified_sample(
        full_calibration_catalogue, CALIBRATION_SIZE, CALIBRATION_SEED,
    )
if len(calibration_catalogue) != CALIBRATION_SIZE:
    raise RuntimeError("Calibration cohort size changed")
sample_ids = set(calibration_catalogue.loc[calibration_catalogue["dataset"].eq(DATASET_NAME), "image_id"])
calibration_role_overlap = {
    role: int(split_frame.loc[split_frame["open_role"].eq(role), "image_id"].isin(sample_ids).sum())
    for role in ("gallery", "query_known", "query_novel")
}
print("Calibration overlap with evaluated roles:", calibration_role_overlap)
calibration_sample_columns = ["dataset", "image_id", "identity"]
calibration_sample_path = OUTPUT_ROOT / CALIBRATION_SAMPLE_NAME
calibration_catalogue[calibration_sample_columns].to_csv(calibration_sample_path, index=False)
print("Wrote calibration cohort:", calibration_sample_path)
# Hash logical membership, not machine-specific absolute image paths or CSV newlines.
calibration_membership = calibration_catalogue[["dataset", "image_id"]].astype(str).to_dict(orient="records")
calibration_sample_sha256 = hashlib.sha256(
    json.dumps(calibration_membership, sort_keys=True, separators=(",", ":")).encode("utf-8")
).hexdigest()
print(calibration_catalogue.groupby("dataset").size())
print("Calibration cohort SHA256:", calibration_sample_sha256)
calibration_catalogue = resolve_catalogue_paths(calibration_catalogue, "calibration")
selected_catalogue = resolve_catalogue_paths(selected_unresolved, DATASET_NAME)

class FrozenStage00Dataset(WildlifeDataset):
    summary = {"animals": ["mixed wildlife"]}

calibration_dataset = FrozenStage00Dataset(root="/", df=calibration_catalogue, check_files=True, load_label=True)
query_catalogue = selected_catalogue.loc[selected_catalogue["open_role"].isin(["query_known", "query_novel"])].reset_index(drop=True)
gallery_catalogue = selected_catalogue.loc[selected_catalogue["open_role"].eq("gallery")].reset_index(drop=True)
query_dataset = FrozenStage00Dataset(root="/", df=query_catalogue, check_files=True, load_label=True)
gallery_dataset = (
    FrozenStage00Dataset(root="/", df=gallery_catalogue, check_files=True, load_label=True)
    if len(gallery_catalogue) else None
)
print("Calibration images:", len(calibration_dataset))
print("Query images:", len(query_catalogue), "identities:", query_catalogue["identity"].nunique())
print("Gallery images:", len(gallery_catalogue), "identities:", gallery_catalogue["identity"].nunique() if len(gallery_catalogue) else 0)

In [ ]:
from apexid.data import build_eval_transforms
from apexid.evaluation.embeddings import ApexIDModelAdapter
from apexid.utils.config import load_config
from apexid.utils.provenance import sha256_file


class NormalizeEmbedding(torch.nn.Module):
    def forward(self, embeddings):
        return torch.nn.functional.normalize(embeddings.float(), dim=1)


class AlbumentationsFromPIL:
    def __init__(self, transform):
        self.transform = transform

    def __call__(self, image: Image.Image):
        return self.transform(image=np.asarray(image.convert("RGB")))["image"]


config = load_config(config_path=MODEL_CONFIG)
if WEIGHTS is not None:
    adapter = ApexIDModelAdapter.from_checkpoint(config, WEIGHTS, DEVICE)
else:
    adapter = ApexIDModelAdapter(config, DEVICE)
    if getattr(adapter.model.backbone, "pretrained_provenance", None) is None:
        raise ValueError("Supply a trained checkpoint or choose a pretrained reference config.")
global_model = torch.nn.Sequential(adapter.model, NormalizeEmbedding())
global_transform = AlbumentationsFromPIL(build_eval_transforms(config=config))
global_provenance = {
    "weights_sha256": adapter.checkpoint_metadata.get("weights_sha256"),
    "config_sha256": sha256_file(MODEL_CONFIG),
    "pretrained_artifact": getattr(adapter.model.backbone, "pretrained_provenance", None),
}
local_transform = T.Compose([T.Resize([512, 512]), T.ToTensor()])

matcher_global = SimilarityPipeline(
    matcher=CosineSimilarity(),
    extractor=DeepFeatures(model=global_model, device=str(DEVICE), batch_size=GLOBAL_BATCH_SIZE),
    transform=global_transform,
    calibration=IsotonicCalibration(),
)
matcher_aliked = SimilarityPipeline(
    matcher=MatchLightGlue(features="aliked", device=str(DEVICE), batch_size=LOCAL_BATCH_SIZE),
    extractor=AlikedExtractor(), transform=local_transform, calibration=IsotonicCalibration(),
)
matcher_disk = SimilarityPipeline(
    matcher=MatchLightGlue(features="disk", device=str(DEVICE), batch_size=LOCAL_BATCH_SIZE, init_threshold=0.9),
    extractor=DiskExtractor(), transform=local_transform, calibration=IsotonicCalibration(),
)
wildfusion = WildFusion(
    calibrated_pipelines=[matcher_aliked, matcher_global, matcher_disk],
    priority_pipeline=matcher_global,
)
print("Global descriptor ready:", GLOBAL_DESCRIPTOR, global_provenance)
print("Calibrating WildFusion on", len(calibration_dataset), "persisted images ...")
print("Calibration cohort SHA256:", calibration_sample_sha256)
wildfusion.fit_calibration(calibration_dataset, calibration_dataset)
print("Calibration complete")


def _finite_clip(matrix, shape, label):
    matrix = np.asarray(matrix, dtype=np.float32)
    matrix[~np.isfinite(matrix)] = 0.0
    matrix = np.clip(matrix, 0.0, 1.0)
    if matrix.shape != shape:
        raise RuntimeError(f"{label} shape {matrix.shape} != {shape}")
    return matrix


def _symmetrize_square(matrix, label):
    max_asymmetry = float(np.max(np.abs(matrix - matrix.T))) if matrix.size else 0.0
    matrix = 0.5 * (matrix + matrix.T)
    np.fill_diagonal(matrix, 1.0)
    print(f"Symmetrized {label}; max |S-S.T| before average: {max_asymmetry:.6f}")
    return matrix


print(
    f"Computing {GLOBAL_DESCRIPTOR}/{DATASET_NAME} query-query: "
    f"{len(query_dataset)} x {len(query_dataset)}"
)
sim_query_query = _finite_clip(
    wildfusion(query_dataset, query_dataset, B=B),
    (len(query_dataset), len(query_dataset)),
    "sim_query_query",
)
sim_query_query = _symmetrize_square(sim_query_query, "query-query")

if gallery_dataset is None:
    sim_query_db = np.zeros((len(query_dataset), 0), dtype=np.float32)
    print("No gallery: query-gallery matrix is empty")
else:
    print(
        f"Computing {GLOBAL_DESCRIPTOR}/{DATASET_NAME} query-gallery: "
        f"{len(query_dataset)} x {len(gallery_dataset)}"
    )
    sim_query_db = _finite_clip(
        wildfusion(query_dataset, gallery_dataset, B=B),
        (len(query_dataset), len(gallery_dataset)),
        "sim_query_db",
    )

query_image_ids = query_catalogue["image_id"].astype(str).to_numpy(dtype="U")
query_identities = query_catalogue["identity"].astype(str).to_numpy(dtype="U")
query_open_role = query_catalogue["open_role"].astype(str).to_numpy(dtype="U")
query_paths = query_catalogue["frozen_path"].astype(str).to_numpy(dtype="U")
db_image_ids = gallery_catalogue["image_id"].astype(str).to_numpy(dtype="U")
db_identities = gallery_catalogue["identity"].astype(str).to_numpy(dtype="U")
db_paths = gallery_catalogue["frozen_path"].astype(str).to_numpy(dtype="U")
bundle = {
    DATASET_NAME: {
        "sim_query_query": sim_query_query,
        "sim_query_db": sim_query_db,
        "query_image_ids": query_image_ids,
        "db_image_ids": db_image_ids,
        "query_identities": query_identities,
        "db_identities": db_identities,
        "query_open_role": query_open_role,
        "query_paths": query_paths,
        "db_paths": db_paths,
        "dataset": DATASET_NAME,
        "global_descriptor": GLOBAL_DESCRIPTOR,
        "protocol": PROTOCOL,
        "matrix_role": "query_query_and_query_gallery",
        "gallery_required": PROTOCOL != "query_only_defaults",
        "split_applied": True,
        "split_path": str(SPLIT_PATH),
        "split_seed": str(query_catalogue["split_seed"].iloc[0]) if "split_seed" in query_catalogue.columns and len(query_catalogue) else "",
        "global_provenance": global_provenance,
        "wildfusion": {
            "local_branches": ["ALIKED+LightGlue", "DISK+LightGlue"],
            "B": B,
            "calibration_images": len(calibration_dataset),
            "calibration_seed": CALIBRATION_SEED,
            "calibration_sample_sha256": calibration_sample_sha256,
            "calibration_role_overlap": calibration_role_overlap,
        },
    }
}
output_path = OUTPUT_ROOT / f"sim_matrix_{GLOBAL_DESCRIPTOR}_{DATASET_NAME}_open.npy"
tmp_path = output_path.with_suffix(".tmp.npy")
np.save(tmp_path, bundle, allow_pickle=True)
tmp_path.replace(output_path)
print("Saved:", output_path, f"({output_path.stat().st_size / 2**30:.2f} GiB)")
loaded = np.load(output_path, allow_pickle=True).item()[DATASET_NAME]
assert loaded["sim_query_query"].shape == sim_query_query.shape
assert loaded["sim_query_db"].shape == sim_query_db.shape
assert np.array_equal(loaded["query_image_ids"].astype(str), query_image_ids.astype(str))
assert np.array_equal(loaded["query_identities"].astype(str), query_identities.astype(str))
if PROTOCOL == "query_only_defaults":
    if sim_query_db.shape[1] != 0:
        raise RuntimeError("Jaguar query-gallery matrix must have zero gallery columns")
    if not np.all(query_open_role == "query_novel"):
        raise RuntimeError("Jaguar queries must all be query_novel")
else:
    if sim_query_db.shape != (len(query_image_ids), len(db_image_ids)):
        raise RuntimeError("Query-gallery shape does not match split roles")
    if set(query_image_ids) & set(db_image_ids):
        raise RuntimeError("Query/gallery image leakage in the 03a artifact")
    novel_identities = set(query_identities[query_open_role == "query_novel"])
    if novel_identities & set(db_identities):
        raise RuntimeError("Novel identities leaked into the gallery matrix")

manifest = {
    "stage": "03a_wildfusion_open_matrices",
    "status": "passed",
    "protocol": PROTOCOL,
    "dataset": DATASET_NAME,
    "global_descriptor": GLOBAL_DESCRIPTOR,
    "matrix_path": output_path.name,
    "query_query_shape": list(sim_query_query.shape),
    "query_gallery_shape": list(sim_query_db.shape),
    "matrix_dtype": str(sim_query_query.dtype),
    "matrix_sha256": hashlib.sha256(output_path.read_bytes()).hexdigest(),
    "matrix_role": "query_query_and_query_gallery",
    "split_applied": True,
    "split_path": str(SPLIT_PATH),
    "gallery_required": PROTOCOL != "query_only_defaults",
    "full_pool_query_only": PROTOCOL == "query_only_defaults",
    "num_query_images": int(len(query_image_ids)),
    "num_gallery_images": int(len(db_image_ids)),
    "num_query_identities": int(pd.Series(query_identities).nunique()),
    "num_gallery_identities": int(pd.Series(db_identities).nunique()) if len(db_identities) else 0,
    "calibration_images": int(len(calibration_dataset)),
    "calibration_scope": "deterministic_dataset_stratified_stage00_subset",
    "calibration_seed": CALIBRATION_SEED,
    "calibration_sample_file": CALIBRATION_SAMPLE_NAME,
    "calibration_sample_sha256": calibration_sample_sha256,
            "calibration_role_overlap": calibration_role_overlap,
    "calibration_dataset_counts": {
        str(name): int(count) for name, count in calibration_catalogue.groupby("dataset").size().items()
    },
    "calibration_selected_dataset_overlap_images": int(calibration_catalogue["dataset"].eq(DATASET_NAME).sum()),
    "calibration_protocol_note": (
        "The fixed calibration cohort may include labelled gallery and query images from the selected dataset."
    ),
    "local_branches": ["ALIKED+LightGlue", "DISK+LightGlue"],
    "B": B,
    "global_provenance": global_provenance,
}
manifest_path = OUTPUT_ROOT / "stage03a_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
print(json.dumps(manifest, indent=2))
print(f"Next: use the matrices in {OUTPUT_ROOT} for 03b.")